# RAG con El Quijote

In [ ]:
import os
from dotenv import load_dotenv

# LangChain Imports
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_pinecone import PineconeVectorStore
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from pinecone import Pinecone, ServerlessSpec

# Carga de variables de entorno (.env)
load_dotenv()

OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")
INDEX_NAME = os.getenv("INDEX_NAME", "rag")
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")

# Inicialización de Pinecone
pc = Pinecone(api_key=PINECONE_API_KEY)

In [ ]:
# 1. Configuración del Modelo de Embeddings
embedding_model = OllamaEmbeddings(
    model="nomic-embed-text-v2-moe", 
    base_url=OLLAMA_BASE_URL, 
    dimensions=512
)

def get_vectorstore():
    return PineconeVectorStore(
        index_name=INDEX_NAME,
        embedding=embedding_model,
    )

In [ ]:
# 2. Indexación (Ingestión de datos)

# Crear el índice en Pinecone si no existe
if INDEX_NAME not in [i.name for i in pc.list_indexes()]:
    print(f"Creando índice: {INDEX_NAME}...")
    pc.create_index(
        name=INDEX_NAME,
        dimension=512,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )

# Cargar el archivo de El Quijote
loader = TextLoader("./data/el_quijote.txt", encoding="utf-8")
documents = loader.load()

# Dividir el texto en chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
)
chunks = splitter.split_documents(documents)

# Subir los chunks a Pinecone (Esto genera los embeddings automáticamente)
vectorstore = PineconeVectorStore.from_documents(
    documents=chunks,
    embedding=embedding_model,
    index_name=INDEX_NAME,
)

print(f"Se han indexado {len(chunks)} fragmentos.")

In [ ]:
# 3. Consulta (Chat con El Quijote)

# Configurar el recuperador (retriever)
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

# Configurar el modelo de lenguaje (LLM) vía Ollama
llm = ChatOllama(model="gemma4:e4b", base_url=OLLAMA_BASE_URL)

# Definir el Prompt
prompt = ChatPromptTemplate.from_template("""
Responde la pregunta usando solo el siguiente contexto:
{context}
Pregunta: {question}
""")

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Crear la cadena (Chain)
chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# Ejemplo de uso
question = "¿Quién es Don Quijote?"
print(f"Pregunta: {question}")
print("Respuesta:", chain.invoke(question))